# Notebook 15 — ChEMBL-Augmented Local CYP QSAR

**Research question:** Do published, chemically similar **human CYP IC50 measurements** improve local structure–activity predictions beyond the **real five-fold Notebook 11 CheMeleon + screening-transfer ensemble**?

**Phase A: provenance-preserving ChEMBL retrieval and analog coverage.** Download CYP-specific IC50 measurements from the ChEMBL REST API, retain assay IDs, quarantine exact OpenADMET molecular overlaps, identify close analogs using RDKit Morgan/Tanimoto similarity, and audit assay-to-assay disagreement.

**Phase B: strictly nested, assay-aware local modeling.** Learn *within-assay* pairwise ΔpIC50 from ChEMBL; separately test absolute ChEMBL kNN after calibration from **outer-/inner-training overlaps only**. Choose correction family and weight only on Notebook 11's authentic **inner OOF** predictions; evaluate on untouched outer folds. Includes a shuffled-ChEMBL negative control.

**Important safeguards:**
- ChEMBL pIC50 is **not** assumed numerically interchangeable with OpenADMET pIC50.
- Only exact, valid human IC50 measurements become regression targets. `<`, `>`, missing units, and invalid records remain in a rejection audit; never turn them into exact targets.
- Exact ChEMBL–OpenADMET molecule matches are **excluded from external reference/pair banks**. They can calibrate an assay *only when that molecule belongs to the current training partition*.
- All external records are public data and may differ in assay conditions, source chronology, and data provenance. This notebook is an external-data extension; check competition disclosure rules and potential source duplication before submitting.
- `FAST_MODE=True` intentionally uses **partial ChEMBL retrieval and incomplete evaluation**. Its results are smoke tests, **not** publishable scores or blind-test submissions.

References: [ChEMBL API](https://chembl.gitbook.io/chembl-interface-documentation/web-services/chembl-data-web-services), [OpenADMET ChEMBL curation demo](https://demos.openadmet.org/en/latest/demos/01_Data_Curation/01_Curate_ChEMBL_Data.html), [challenge data](https://huggingface.co/datasets/openadmet/cyp-challenge-train-test).

## 0 · Install (Google Colab CPU)

The real ChEMBL download is potentially large: cached API pages are saved individually. For full retrieval, use a Google Drive directory as `PERSISTENT_ROOT`. The experiment reuses existing Notebook 11 predictions and **does not need a GPU or a CheMeleon checkpoint**.

In [ ]:
%pip -q install "rdkit>=2023.9" "scikit-learn>=1.5" "requests>=2.31" "pandas>=2.0" "numpy>=1.26" "scipy>=1.11" matplotlib

## 1 · Configuration

Upload your **real** `cyp_notebook_11_results.zip` to Colab, or set its Google Drive path. Run with `FAST_MODE=True` for a smoke test; then `FAST_MODE=False` using the *same persistent cache root*. All four CYP isoforms are downloaded independently. You can also supply four locally cached raw ChEMBL CSV files via `CHEMBL_RAW_OVERRIDE` (must be an explicit mapping by CYP name).

In [ ]:
from pathlib import Path
import os, io, json, hashlib, zipfile, math, time, warnings
from collections import defaultdict
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs
from rdkit import RDLogger
from rdkit.Chem import rdFingerprintGenerator
from sklearn.linear_model import Ridge
from IPython.display import display

RDLogger.DisableLog('rdApp.warning')

FAST_MODE = True
NB11_RESULTS_ZIP = '/content/cyp_notebook_11_results.zip'
PERSISTENT_ROOT = '/content/cyp_nb15'
# For Drive persistence, e.g. /content/drive/MyDrive/CYP/Notebook15
# from google.colab import drive; drive.mount('/content/drive')
CHEMBL_RAW_OVERRIDE = {}  # {'CYP1A2':'/content/cyp1a2_activity.csv', ... all 4}
PAGE_LIMIT = 1000
MAX_FAST_PAGES_PER_ISO = 2
MIN_NEIGHBOR_SIM = 0.55
MIN_PAIR_SIM = 0.52
N_FP_BITS = 512
MAX_PAIRS_PER_ISO = 1200 if FAST_MODE else 6000
MAX_EVAL_ROWS_PER_ISO = 75 if FAST_MODE else None
N_BOOTSTRAP = 40 if FAST_MODE else 500
BLEND_STRENGTHS = (0.00, 0.15, 0.30, 0.50)
SEED = 2026
EXPERIMENT_TAG = 'chembl_augmented_v3'
RUN_SHUFFLE_CONTROL = True
BLIND_SMILES_CSV = None  # Optional: official blind structure file; structure-only audit + exact-match quarantine
CHEMBL_SOURCE_COMPLETE_CONFIRMED = False  # Set True ONLY if your supplied raw CSVs are complete target exports

BASE = Path(PERSISTENT_ROOT)
BASE.mkdir(parents=True,exist_ok=True)
CACHE = BASE/'cache'
CACHE.mkdir(exist_ok=True)
OUT = BASE/'results'
OUT.mkdir(exist_ok=True)

if not Path(NB11_RESULTS_ZIP).exists():
    for p in [Path.cwd()/'cyp_notebook_11_results.zip',Path('/mnt/data/cyp_notebook_11_results.zip')]:
        if p.exists(): NB11_RESULTS_ZIP=str(p);break
if not Path(NB11_RESULTS_ZIP).exists():
    raise FileNotFoundError('Upload full real cyp_notebook_11_results.zip and set NB11_RESULTS_ZIP.')
FOLDS_TO_RUN = [0] if FAST_MODE else list(range(5))
print('Notebook 15 mode:', 'PARTIAL SMOKE TEST' if FAST_MODE else 'FULL RESEARCH RUN')
print('Notebook 11 input:',NB11_RESULTS_ZIP)

## 2 · Self-contained functions

The next cell embeds the complete extraction/curation/modeling implementation so this `.ipynb` runs without cloning the repository. A matching copy also lives in `src/chembl_local_qsar.py` for unit testing.

In [ ]:
"""Notebook 15 self-contained implementation. ChEMBL data retrieval and nested local SAR.

Outputs are diagnostic; ChEMBL IC50 values should not be pooled directly with OpenADMET.
"""
import hashlib
import io
import json
import math
import os
import time
import warnings
import zipfile
from collections import defaultdict
from pathlib import Path
from urllib.parse import urljoin

import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import AllChem, Descriptors, rdFingerprintGenerator, rdMolDescriptors
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

ISOFORMS = ['CYP1A2','CYP2C9','CYP2D6','CYP3A4']
TARGET_IDS = {'CYP1A2':'CHEMBL3356','CYP2C9':'CHEMBL3397',
              'CYP2D6':'CHEMBL289','CYP3A4':'CHEMBL340'}
API = 'https://www.ebi.ac.uk/chembl/api/data/'
DATA_FIELDS = ['activity_id','molecule_chembl_id','canonical_smiles','target_chembl_id',
               'target_organism','standard_type','standard_value','standard_units',
               'standard_relation','pchembl_value','assay_chembl_id','assay_type',
               'bao_label','document_chembl_id','data_validity_comment','potential_duplicate']
DESC_NAMES = ['MolWt','MolLogP','TPSA','NumHDonors','NumHAcceptors','NumRotatableBonds',
              'RingCount','FractionCSP3']


def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1<<20),b''):
            h.update(block)
    return h.hexdigest()


def sha256_frame(df, cols):
    subset=df[cols].copy().sort_values(cols,kind='mergesort').reset_index(drop=True)
    return hashlib.sha256(subset.to_csv(index=False,na_rep='NA').encode('utf-8')).hexdigest()


def canonical(s):
    if not isinstance(s,str) or not s.strip(): return None
    m=Chem.MolFromSmiles(s)
    return Chem.MolToSmiles(m, canonical=True, isomericSmiles=True) if m else None


def open_nb11(path):
    path=Path(path)
    if not path.is_file(): raise FileNotFoundError(f'Upload the original Notebook 11 ZIP: {path}')
    with zipfile.ZipFile(path) as z:
        def load(name):
            paths=[p for p in z.namelist() if p==name or p.endswith('/'+name)]
            if len(paths)!=1: raise ValueError(f'Expected exactly one {name}: {paths}')
            return z.read(paths[0])
        meta=json.loads(load('run_metadata.json'))
        if not(meta.get('complete_five_fold_oof') and meta.get('scientifically_valid_full_model_run')):
            raise RuntimeError('Notebook 11 results are partial or synthetic; do not use as real baseline.')
        if meta.get('graph_proxy_test_only') or not meta.get('graph'):
            raise RuntimeError('Notebook 11 used graph proxy rather than fine-tuned CheMeleon.')
        outer=pd.read_csv(io.BytesIO(load('nested_outer_predictions.csv')))
        inner=pd.read_csv(io.BytesIO(load('inner_oof_predictions_for_selection.csv')))
        manifest=pd.read_csv(io.BytesIO(load('fold_manifest.csv')))
    model='CheMeleon_finetuned + ScreenStack_LGBM'
    outer=outer[outer.subset.eq(model)].copy()
    if len(outer)!=6525 or len(manifest)!=4905:
        raise ValueError(f'Unexpected Notebook 11 coverage: outer={len(outer)} manifest={len(manifest)}')
    if outer.duplicated(['fold','canonical_smiles','isoform']).any():
        raise ValueError('Duplicated outer predictions')
    if inner.duplicated(['fold','canonical_smiles','isoform']).any():
        raise ValueError('Duplicated inner predictions')
    manifest.fold=manifest.fold.astype(int)
    allfold=manifest.set_index('canonical_smiles').fold
    if not (outer.fold.astype(int).to_numpy()==outer.canonical_smiles.map(allfold).to_numpy()).all():
        raise AssertionError('Notebook 11 outer-fold assignment mismatch')
    if not set(outer.isoform)==set(ISOFORMS): raise ValueError('Missing CYP endpoint')
    inner['baseline']=0.5*(inner.CheMeleon_finetuned+inner.ScreenStack_LGBM)
    outer=outer.rename(columns={'y_pred':'baseline'})
    for c in ['canonical_smiles','scaffold_group','isoform']:
        outer[c]=outer[c].astype(str)
    if not outer[['y_true','baseline','ci_low','ci_high']].notna().all().all():
        raise AssertionError('OOF data has missing bounds/prediction')
    ci=outer[['canonical_smiles','isoform','ci_low','ci_high']].drop_duplicates(['canonical_smiles','isoform'])
    inner=inner.merge(ci,on=['canonical_smiles','isoform'],how='left',validate='many_to_one')
    if not inner[['ci_low','ci_high','baseline']].notna().all().all():
        raise AssertionError('Inner validation CI mapping failed')
    return meta, outer, inner, manifest


def session_with_retries():
    import requests
    from requests.adapters import HTTPAdapter
    from urllib3.util.retry import Retry
    retry=Retry(total=5,connect=5,read=5,backoff_factor=1.5,
                status_forcelist=[429,500,502,503,504],
                allowed_methods=frozenset(['GET']))
    s=requests.Session()
    s.mount('https://',HTTPAdapter(max_retries=retry,pool_connections=4,pool_maxsize=4))
    s.headers.update({'User-Agent':'OpenADMET-CYP-Notebook15-research/1.0'})
    return s


def request_json(session,url,params=None):
    response=session.get(url,params=params,timeout=(15,120))
    response.raise_for_status()
    d=response.json()
    if not isinstance(d,dict):raise ValueError('ChEMBL API returned unexpected non-JSON-object')
    return d


def download_chembl_pages(cache_dir,isoform,fast_mode=False,page_limit=1000,max_pages_fast=2,
                          pause=0.2):
    """Download target+IC50 records with one persistent file per page.

    Fails closed on network errors. FAST_MODE intentionally downloads only a prefix.
    """
    cache_dir=Path(cache_dir)/isoform
    cache_dir.mkdir(parents=True,exist_ok=True)
    target=TARGET_IDS[isoform]
    params={'target_chembl_id':target,'standard_type':'IC50','limit':page_limit}
    session=session_with_retries()
    offset=0; total=None; pages=0; parts=[]
    while True:
        params['offset']=offset
        pagefile=cache_dir/f'{target}_IC50_limit{page_limit}_offset{offset:08d}.json'
        if pagefile.exists():
            data=json.loads(pagefile.read_text())
        else:
            data=request_json(session,API+'activity.json',params=params)
            if not {'activities','page_meta'} <= data.keys():
                raise RuntimeError('ChEMBL response schema unexpected, aborting cache')
            tmp=pagefile.with_suffix('.json.partial')
            tmp.write_text(json.dumps(data))
            os.replace(tmp,pagefile)
            time.sleep(pause)
        vals=data.get('activities',[]); meta=data.get('page_meta',{})
        if total is None: total=int(meta['total_count'])
        if int(meta['offset'])!=offset: raise AssertionError('Cached ChEMBL page has wrong offset')
        if int(meta['total_count'])!=total:raise RuntimeError('ChEMBL dataset changed during pagination. Use a new cache folder.')
        parts.extend(vals); pages+=1
        offset+=len(vals)
        if len(vals)==0 or offset>=total: break
        if fast_mode and pages>=max_pages_fast: break
        if len(vals)>page_limit:raise AssertionError('API page exceeds request limit')
    complete=(offset>=total)
    frame=pd.DataFrame(parts)
    for col in DATA_FIELDS:
        if col not in frame:frame[col]=np.nan
    return frame[DATA_FIELDS],{'isoform':isoform,'target_chembl_id':target,
        'received':len(frame),'expected':total,'pages':pages,'complete':complete}


def normalize_activities(raw,isoform):
    """Reversible, exact human IC50 only; preserve other records in a rejected audit."""
    r=raw.copy()
    r['isoform']=isoform
    for k in DATA_FIELDS:
        if k not in r:r[k]=np.nan
    r['reject_reason']=''
    def reject(mask,why):
        mask=np.asarray(mask,dtype=bool)
        r.loc[(r.reject_reason=='')&mask,'reject_reason']=why
    reject(r.target_chembl_id.astype(str).ne(TARGET_IDS[isoform]),'wrong_target')
    reject(r.standard_type.astype(str).str.upper().ne('IC50'),'not_IC50')
    org=r.target_organism.fillna('').astype(str).str.lower()
    reject(org.ne('') & org.ne('homo sapiens'),'not_human')
    relations=r.standard_relation.fillna('').astype(str).str.strip()
    reject(relations.ne('='), 'censored_or_missing_relation')
    units=r.standard_units.fillna('').astype(str).str.lower().str.replace('µ','u',regex=False).str.replace('μ','u',regex=False)
    mult=units.map({'nm':1e-9,'um':1e-6,'mm':1e-3,'pm':1e-12,'m':1.0})
    reject(mult.isna(),'invalid_units')
    v=pd.to_numeric(r.standard_value,errors='coerce')
    reject(~np.isfinite(v)|(v<=0),'nonpositive_or_missing_value')
    validity=r.data_validity_comment.fillna('').astype(str).str.strip()
    reject(validity.ne('') & validity.ne('Manually validated'),'data_validity_flag')
    duplicate=r.potential_duplicate.astype('string').fillna('false').str.lower().isin(['true','1','yes'])
    reject(duplicate,'potential_duplicate')
    r['canonical_smiles']=r.canonical_smiles.map(canonical)
    reject(r.canonical_smiles.isna(),'invalid_structure')
    r['pIC50']=-np.log10(v*mult)
    reject(~np.isfinite(r.pIC50)|~r.pIC50.between(1.0,13.0),'implausible_pIC50')
    r['assay_chembl_id']=r.assay_chembl_id.fillna('UNKNOWN').astype(str)
    r['molecule_chembl_id']=r.molecule_chembl_id.fillna('').astype(str)
    clean=r[r.reject_reason.eq('')].copy()
    if clean.activity_id.notna().any():
        identified=clean[clean.activity_id.notna()].drop_duplicates(['activity_id'])
        unidentified=clean[clean.activity_id.isna()]
        clean=pd.concat([identified,unidentified],ignore_index=True)
    discarded=r[r.reject_reason.ne('')].copy()
    return clean,discarded


def aggregate_per_assay(clean):
    """Do not mix assay measurements; aggregate duplicates within assay+compound only."""
    groups=['isoform','assay_chembl_id','canonical_smiles']
    if clean.empty:return pd.DataFrame(columns=groups+['pIC50','n_records','assay_mad','molecule_chembl_id','document_chembl_id'])
    return (clean.groupby(groups,sort=True).agg(
        pIC50=('pIC50','median'),n_records=('pIC50','size'),
        assay_mad=('pIC50',lambda v:float(np.median(np.abs(v-np.median(v))))),
        molecule_chembl_id=('molecule_chembl_id','first'),
        document_chembl_id=('document_chembl_id','first'))
        .reset_index())


def morgan_gen(n_bits=512):
    return rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=int(n_bits),includeChirality=True)


def fp_for(s, generator):
    m=Chem.MolFromSmiles(s)
    if m is None:raise ValueError('Invalid molecule '+str(s))
    return generator.GetFingerprint(m)


def descriptor_for(s):
    m=Chem.MolFromSmiles(s)
    if m is None:raise ValueError('Invalid molecule '+str(s))
    return np.array([Descriptors.MolWt(m),Descriptors.MolLogP(m),Descriptors.TPSA(m),
        rdMolDescriptors.CalcNumHBD(m),rdMolDescriptors.CalcNumHBA(m),
        rdMolDescriptors.CalcNumRotatableBonds(m),rdMolDescriptors.CalcNumRings(m),
        rdMolDescriptors.CalcFractionCSP3(m)],dtype=np.float32)


def feature_for(s, generator):
    fp=fp_for(s,generator)
    bits=np.zeros((generator.GetOptions().fpSize,),dtype=np.float32)
    DataStructs.ConvertToNumpyArray(fp,bits)
    desc=descriptor_for(s)
    return np.r_[bits,desc].astype(np.float32)


def metric_mae(frame,col):
    vals=[]
    for iso in ISOFORMS:
        f=frame[(frame.isoform==iso) & frame[col].notna()]
        if len(f)==0:return np.inf
        vals.append(float(np.abs(f[col]-f.y_true).mean()))
    return float(np.mean(vals))


def st_rae(frame,col):
    scores=[]
    for iso in ISOFORMS:
        f=frame[(frame.isoform==iso) & frame[col].notna()]
        if len(f)<2: return np.inf
        y=f.y_true.to_numpy(float); yp=f[col].to_numpy(float)
        lo=f.ci_low.to_numpy(float);hi=f.ci_high.to_numpy(float)
        if not(np.isfinite(np.c_[y,yp,lo,hi]).all() and (lo<=hi).all()):return np.inf
        loss=np.maximum(lo-yp,0)+np.maximum(yp-hi,0)
        mu=float(y.mean());baseline=np.maximum(lo-mu,0)+np.maximum(mu-hi,0)
        denom=baseline.sum()
        if denom<=1e-12:return np.inf
        scores.append(float(loss.sum()/denom))
    return float(np.mean(scores))


def max_similarity(query_smi, ref_smis, gen, fps=None):
    if not ref_smis:return 0.,None
    refs=fps if fps is not None else [fp_for(s,gen) for s in ref_smis]
    sims=np.asarray(DataStructs.BulkTanimotoSimilarity(fp_for(query_smi,gen),refs))
    j=int(np.argmax(sims))
    return float(sims[j]),ref_smis[j]


def nearest_cached(query_smi, library, fp_cache, top_k=5):
    """library: list of (canonical_smiles, assay, y) with fp_cache keyed by smiles."""
    if not library:return []
    qfp=fp_cache[query_smi]
    f=[fp_cache[x[0]] for x in library]
    sims=np.array(DataStructs.BulkTanimotoSimilarity(qfp,f),dtype=np.float32)
    k=min(top_k,len(sims))
    ix=np.argpartition(-sims,k-1)[:k]
    ix=sorted(ix,key=lambda i:(-sims[i],library[i][0],library[i][1]))
    return [(float(sims[i]),library[i]) for i in ix]


def pair_delta_features(s_q,s_a,feat_cache):
    x=feat_cache[s_q]-feat_cache[s_a]
    return x


def create_chmbl_delta_pairs(external, fp_cache, feat_cache,isoform,
                              min_sim=.52,k_neigh=4,max_pairs=5500,seed=2026):
    """Only within identical ChEMBL assay IDs to reduce scale differences."""
    e=external[external.isoform.eq(isoform)].copy()
    rng=np.random.default_rng(seed)
    pairs=[]
    for assay,group in e.groupby('assay_chembl_id'):
        group=group.drop_duplicates('canonical_smiles').reset_index(drop=True)
        if len(group)<2:continue
        if len(group)>250:
            group=group.iloc[np.sort(rng.choice(len(group),250,replace=False))].reset_index(drop=True)
        smis=group.canonical_smiles.tolist()
        fps=[fp_cache[s] for s in smis]
        for i,s in enumerate(smis):
            sims=np.array(DataStructs.BulkTanimotoSimilarity(fps[i],fps))
            sims[i]=-1
            good=np.where(sims>=min_sim)[0]
            if not len(good):continue
            best=good[np.argsort(sims[good])[::-1][:k_neigh]]
            for j in best:
                if i<j:
                    pairs.append((s,smis[j],float(group.pIC50.iloc[i]-group.pIC50.iloc[j]),assay))
        if len(pairs)>max_pairs*3: break
    if len(pairs)>max_pairs:
        sel=np.sort(rng.choice(len(pairs),max_pairs,replace=False));pairs=[pairs[i] for i in sel]
    if len(pairs)<12: return None,pd.DataFrame(pairs,columns=['query','anchor','delta','assay_id'])
    X=np.asarray([pair_delta_features(a,b,feat_cache) for a,b,_,_ in pairs],dtype=np.float32)
    y=np.asarray([d for _,_,d,_ in pairs],dtype=np.float32)
    X=np.concatenate([X,-X],axis=0)
    y=np.concatenate([y,-y])
    scaler=StandardScaler(with_mean=False) # sparse-like fingerprint diffs + small descriptors
    model=make_pipeline(scaler,Ridge(alpha=250.0,fit_intercept=False,solver='lsqr'))
    model.fit(X,y)
    pframe=pd.DataFrame(pairs,columns=['query','anchor','delta','assay_id'])
    return model,pframe


def make_calibration(overlap, primary_train, iso, min_assay_overlaps=2,min_global_overlaps=3):
    """Activity calibration exclusively from outer/inner training molecule matches."""
    match=overlap[(overlap.isoform==iso)].merge(
        primary_train[primary_train.isoform.eq(iso)][['canonical_smiles','y_true']],
        on='canonical_smiles',how='inner',validate='many_to_one')
    if len(match): match=match.assign(offset=match.y_true-match.pIC50)
    by_assay={}
    for assay,g in match.groupby('assay_chembl_id'):
        if g.canonical_smiles.nunique()>=min_assay_overlaps:
            by_assay[str(assay)]=float(g.groupby('canonical_smiles').offset.median().median())
    global_offset=(float(match.groupby('canonical_smiles').offset.median().median())
                   if match.canonical_smiles.nunique()>=min_global_overlaps else np.nan)
    return by_assay,global_offset,int(match.canonical_smiles.nunique())


def calc_external_neighbors(smi,iso,external_by_iso,fp_cache,ext_neighbor_cache,k=5):
    key=(smi,iso)
    if key in ext_neighbor_cache:return ext_neighbor_cache[key]
    lib=external_by_iso.get(iso,[])
    near=nearest_cached(smi,lib,fp_cache,top_k=k)
    ext_neighbor_cache[key]=near
    return near


def calc_local_one(qsmi,iso,training,delta_model,external_by_iso,fp_cache,
                   feat_cache,ext_neighbor_cache,calib,min_ext_sim=.55,
                   min_primary_sim=.35,max_primary_neighbors=3,
                   training_index=None, negative_model=None):
    """No held-out y or in-fold OOF residual is ever used as an anchor."""
    if training_index is None:
        refs=training[training.isoform.eq(iso)].drop_duplicates('canonical_smiles')
        refs=refs[refs.canonical_smiles.ne(qsmi)]
        refsmis=refs.canonical_smiles.astype(str).tolist()
        refy=refs.y_true.to_numpy(float)
        reffps=[fp_cache[x] for x in refsmis]
    else:
        refsmis,reffps,refy=training_index
    if not refsmis:return {'delta_local':np.nan,'delta_gate':0.,'calib_local':np.nan,
                          'calib_gate':0.,'primary_max_sim':0.,'ext_max_sim':0.,'n_calibrated_neighbors':0,
                          'negative_delta_local':np.nan}
    sims=np.asarray(DataStructs.BulkTanimotoSimilarity(fp_cache[qsmi],reffps),dtype=float)
    if qsmi in refsmis: raise AssertionError('Query compound was present in reference anchor set')
    count=min(max_primary_neighbors,len(sims))
    loc=np.argpartition(-sims,count-1)[:count]
    loc=sorted(loc,key=lambda i:-sims[i]);mx=float(sims[loc[0]])
    ext_near=calc_external_neighbors(qsmi,iso,external_by_iso,fp_cache,ext_neighbor_cache)
    extmax=float(ext_near[0][0]) if ext_near else 0.
    primary_gate=float(np.clip((mx-min_primary_sim)/.35,0,1))
    ext_gate=float(np.clip((extmax-min_ext_sim)/.30,0,1))
    delta_local=np.nan;negative_local=np.nan
    if delta_model is not None or negative_model is not None:
        anchor_smis=[refsmis[i] for i in loc]
        X=np.asarray([pair_delta_features(qsmi,s,feat_cache) for s in anchor_smis])
        w=np.asarray([float(max(sims[i],.01)**4) for i in loc])
        if delta_model is not None:
            deltas=np.asarray(delta_model.predict(X))
            delta_local=float(np.average(np.asarray([float(refy[i])+d for i,d in zip(loc,deltas)]),weights=w))
        if negative_model is not None:
            neg=np.asarray(negative_model.predict(X))
            negative_local=float(np.average(np.asarray([float(refy[i])+d for i,d in zip(loc,neg)]),weights=w))
    delta_gate=primary_gate*ext_gate if np.isfinite(delta_local) else 0.
    by_assay,glob,n_overlap=calib
    p=[];w=[]
    for sim,(s,assay,y) in ext_near:
        if sim<min_ext_sim:continue
        offset=by_assay.get(str(assay),glob)
        if not np.isfinite(offset):continue
        p.append(float(y+offset));w.append(float(sim**5))
    calibrated=float(np.average(p,weights=w)) if p else np.nan
    cal_gate=ext_gate*min(1.0,len(p)/3) if p else 0.
    return {'delta_local':delta_local,'delta_gate':delta_gate,
            'calib_local':calibrated,'calib_gate':cal_gate,
            'primary_max_sim':mx,'ext_max_sim':extmax,'n_calibrated_neighbors':len(p),
            'n_calibration_overlap':n_overlap,'negative_delta_local':negative_local}


def add_candidate_predictions(frame,strengths=(0.,.15,.30,.50)):
    out=frame.copy()
    for name in ['delta','calib','joint']:
        for v in strengths:
            key=f'{name}_{round(v,2):.2f}'
            if name=='delta': corr=np.where(np.isfinite(out.delta_local),
                        out.delta_gate*(out.delta_local-out.baseline),0.)
            elif name=='calib': corr=np.where(np.isfinite(out.calib_local),
                        out.calib_gate*(out.calib_local-out.baseline),0.)
            else:
                d=np.where(np.isfinite(out.delta_local),out.delta_gate*(out.delta_local-out.baseline),0.)
                c=np.where(np.isfinite(out.calib_local),out.calib_gate*(out.calib_local-out.baseline),0.)
                corr=.5*d+.5*c
            out[key]=(out.baseline+v*corr).astype(float)
    return out


def scored_candidates(frame,strengths=(0.,.15,.30,.50),metric='st_rae'):
    fn=st_rae if metric=='st_rae' else metric_mae
    keys=['baseline']+[f'{fam}_{v:.2f}' for fam in ('delta','calib','joint') for v in strengths if v>0]
    scores=[]
    for key in keys:
        scores.append({'candidate':key,'macro_st_rae':st_rae(frame,key),
                       'macro_mae':metric_mae(frame,key)})
    return pd.DataFrame(scores).sort_values(['macro_st_rae','macro_mae','candidate']).reset_index(drop=True)


def paired_group_bootstrap(frame,baseline='baseline',candidate='selected',group='scaffold_group',
                           reps=300,seed=2026):
    """Paired molecule-scaffold bootstrap; recomputes ST-RAE normalization in each draw."""
    rng=np.random.default_rng(seed)
    unique_groups, group_index=np.unique(frame[group].astype(str).to_numpy(),return_inverse=True)
    n_groups=len(unique_groups)
    iso=frame.isoform.astype(str).to_numpy()
    y=frame.y_true.to_numpy(float)
    lo=frame.ci_low.to_numpy(float); hi=frame.ci_high.to_numpy(float)
    base=frame[baseline].to_numpy(float); cand=frame[candidate].to_numpy(float)
    records=[]
    for i in range(reps):
        group_weight=rng.multinomial(n_groups,np.ones(n_groups)/n_groups)
        w=group_weight[group_index]
        changes=[]; st_changes=[]
        for one in ISOFORMS:
            mask=(iso==one) & (w>0)
            weights=w[mask].astype(float)
            if weights.sum()<=1:continue
            truth=y[mask]; yy0=base[mask]; yy1=cand[mask]
            avg=np.average(truth,weights=weights)
            denom=np.dot(weights,np.maximum(lo[mask]-avg,0)+np.maximum(avg-hi[mask],0))
            if denom<1e-12:continue
            b_loss=np.maximum(lo[mask]-yy0,0)+np.maximum(yy0-hi[mask],0)
            c_loss=np.maximum(lo[mask]-yy1,0)+np.maximum(yy1-hi[mask],0)
            changes.append(np.average(abs(truth-yy0)-abs(truth-yy1),weights=weights))
            st_changes.append((np.dot(weights,b_loss)-np.dot(weights,c_loss))/denom)
        if len(changes)==len(ISOFORMS):
            records.append({'bootstrap':i,'mae_gain':float(np.mean(changes)),
                           'st_rae_gain':float(np.mean(st_changes))})
    return pd.DataFrame(records)

## 3 · Load and validate the authentic Notebook 11 nested predictions

The run **fails closed** if the source is a proxy, incomplete or otherwise not the real five-fold experiment. Confidence intervals for the inner selection set come from *training-compound assay metadata*, not any held-out model predictions.

In [ ]:
NB11_META, OUTER, INNER, MANIFEST = open_nb11(NB11_RESULTS_ZIP)
NB11_SHA = sha256_file(NB11_RESULTS_ZIP)
ALL_PRIMARY = set(MANIFEST.canonical_smiles)
SCAFFOLD_BY_SMILES=MANIFEST.set_index('canonical_smiles').scaffold_group.to_dict()
ALL_LABELS = OUTER[['canonical_smiles','isoform','y_true']].drop_duplicates(['canonical_smiles','isoform']).copy()
assert len(ALL_LABELS)==6525
assert set(INNER.fold.unique()) == set(range(5))
print('Real five-fold base predictions:',len(OUTER),'measurements,',len(MANIFEST),'structures')
print('Source SHA256:',NB11_SHA)
print('Reference two-model ensemble macro MAE:',round(metric_mae(OUTER,'baseline'),6))
print('Reference two-model ensemble macro ST-RAE:',round(st_rae(OUTER,'baseline'),6))

## 4 · Retrieve ChEMBL IC50 by human CYP target, with resumable page cache

Uses the ChEMBL `/activity.json` API with the established human targets: CYP1A2 `CHEMBL3356`, CYP2C9 `CHEMBL3397`, CYP2D6 `CHEMBL289`, CYP3A4 `CHEMBL340`. Fetching per target is more efficient than running one chemical similarity API request per OpenADMET compound.

**Partial downloads never masquerade as complete datasets.** If the API fails mid-run, successful pages remain cached. The next run resumes them. For the final experiment `FAST_MODE=False` requests every available page, then verifies `received == expected`.

In [ ]:
RAW_FRAMES = []
RETRIEVAL_AUDIT = []
if CHEMBL_RAW_OVERRIDE and set(CHEMBL_RAW_OVERRIDE)!=set(ISOFORMS):
    raise ValueError('CHEMBL_RAW_OVERRIDE must provide all four isoforms.')
for iso in ISOFORMS:
    if CHEMBL_RAW_OVERRIDE:
        path=Path(CHEMBL_RAW_OVERRIDE[iso])
        if not path.is_file():raise FileNotFoundError(path)
        frame=pd.read_csv(path)
        record={'isoform':iso,'target_chembl_id':TARGET_IDS[iso],
                'received':len(frame),'expected':len(frame),'pages':None,
                'complete':True,'source':'user_csv','source_sha256':sha256_file(path)}
    else:
        frame,record=download_chembl_pages(CACHE/'chembl_activity_pages',iso,
                            fast_mode=FAST_MODE,page_limit=PAGE_LIMIT,
                            max_pages_fast=MAX_FAST_PAGES_PER_ISO)
        record['source']='chembl_rest_api'
    frame['source_isoform']=iso
    RAW_FRAMES.append(frame)
    RETRIEVAL_AUDIT.append(record)
    print(iso,'records:',len(frame),'complete:',record['complete'])
RAW=pd.concat(RAW_FRAMES,ignore_index=True)
AUDIT_RETRIEVAL=pd.DataFrame(RETRIEVAL_AUDIT)
AUDIT_RETRIEVAL.to_csv(OUT/'chembl_retrieval_audit.csv',index=False)
RETRIEVAL_COMPLETE=bool(AUDIT_RETRIEVAL.complete.all())
if not RETRIEVAL_COMPLETE:
    print('WARNING: PARTIAL TARGET DOWNLOADS — EXPERIMENTAL RESULTS ARE ONLY A SMOKE TEST')
RAW.to_csv(CACHE/'chembl_activity_raw_merged.csv',index=False)

## 5 · Curation, provenance, and overlap quarantine

Accepted records must have a valid SMILES, correct human CYP target, `standard_type=IC50`, exact `standard_relation` (`=`), known molar units, positive potency and acceptable validity/duplicate flags. All other activities are audited, **not coerced into exact pIC50 labels**.

Measurements are aggregated **only within the same assay and canonical structure**, so assay-specific offsets are not averaged away. ChEMBL molecules identical to **any** OpenADMET primary structure are quarantined: they are not included in external analog predictions or pairwise training. For assay calibration, these exact matches may only be accessed in the current training partition.

In [ ]:
CLEAN=[];REJECTED=[]
for iso,frame in zip(ISOFORMS,RAW_FRAMES):
    clean,rej=normalize_activities(frame,iso)
    CLEAN.append(clean);REJECTED.append(rej)
    print(iso,'accepted:',len(clean),'rejected:',len(rej),
          'distinct compounds:',clean.canonical_smiles.nunique())
CLEAN=pd.concat(CLEAN,ignore_index=True)
REJECTED=pd.concat(REJECTED,ignore_index=True)
ASSAY_MEASUREMENTS=aggregate_per_assay(CLEAN)
ASSAY_MEASUREMENTS['openadmet_overlap'] = ASSAY_MEASUREMENTS.canonical_smiles.isin(ALL_PRIMARY)
BLIND_EXACT = set()
if BLIND_SMILES_CSV:
    blind_struct=pd.read_csv(BLIND_SMILES_CSV)
    if 'SMILES' not in blind_struct: raise KeyError('Blind audit requires SMILES column')
    if any('pIC50' in c or 'IC50' in c for c in blind_struct.columns):
        raise ValueError('Only BLINDED STRUCTURES are permitted; remove activity labels')
    BLIND_EXACT = set(blind_struct.SMILES.map(canonical).dropna())
ASSAY_MEASUREMENTS['blind_exact_match'] = ASSAY_MEASUREMENTS.canonical_smiles.isin(BLIND_EXACT)
BLIND_QUARANTINE=ASSAY_MEASUREMENTS[ASSAY_MEASUREMENTS.blind_exact_match].copy()
BLIND_QUARANTINE.to_csv(OUT/'chembl_quarantined_blind_exact_matches.csv',index=False)

OVERLAP=ASSAY_MEASUREMENTS[ASSAY_MEASUREMENTS.openadmet_overlap & ~ASSAY_MEASUREMENTS.blind_exact_match].copy()
EXTERNAL=ASSAY_MEASUREMENTS[~ASSAY_MEASUREMENTS.openadmet_overlap & ~ASSAY_MEASUREMENTS.blind_exact_match].copy()
assert not EXTERNAL.canonical_smiles.isin(ALL_PRIMARY).any()
assert not set(EXTERNAL.canonical_smiles)&ALL_PRIMARY
assert not set(EXTERNAL.canonical_smiles)&BLIND_EXACT
for name,df in [('chembl_curated_assay_measurements',ASSAY_MEASUREMENTS),
                ('chembl_excluded_exact_overlaps',OVERLAP),
                ('chembl_external_references',EXTERNAL)]:
    df.to_csv(OUT/(name+'.csv'),index=False,float_format='%.10g')
if len(REJECTED):
    REJECTED[['isoform','activity_id','molecule_chembl_id','reject_reason']].to_csv(
        OUT/'chembl_rejected_measurements.csv',index=False)
REJECTION_SUMMARY=REJECTED.groupby(['isoform','reject_reason']).size().reset_index(name='n')
REJECTION_SUMMARY.to_csv(OUT/'chembl_rejection_summary.csv',index=False)
PROVENANCE=ASSAY_MEASUREMENTS.groupby('isoform').agg(
    records=('n_records','sum'),unique_molecules=('canonical_smiles','nunique'),
    assays=('assay_chembl_id','nunique'),documents=('document_chembl_id','nunique')).reset_index()
PROVENANCE.to_csv(OUT/'chembl_provenance_counts.csv',index=False)
print('External usable assay/structure observations:',len(EXTERNAL),
      '| exact OpenADMET overlap observations (calibration only):',len(OVERLAP))
display(PROVENANCE)

## 6 · Chemical similarity audit and assay disagreement

Compute exact 512-bit Morgan radius-2 Tanimoto similarities consistently on both sides. This is intentionally a **locally defined fingerprint**; it need not be numerically identical to the ChEMBL similarity service. The default full audit measures CYP-specific coverage and records assay-level disagreement for structures measured in multiple assays. Optionally audit the unlabeled challenge test SMILES without touching withheld labels.

In [ ]:
GEN=rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=N_FP_BITS)
STRUCTURES=sorted(set(ALL_PRIMARY)|set(ASSAY_MEASUREMENTS.canonical_smiles))
FP_CACHE={s:fp_for(s,GEN) for s in STRUCTURES}
FEAT_CACHE={s:feature_for(s,GEN) for s in STRUCTURES}
EXTERNAL_BY_ISO={}
for iso in ISOFORMS:
    g=EXTERNAL[EXTERNAL.isoform==iso]
    EXTERNAL_BY_ISO[iso]=[(str(r.canonical_smiles),str(r.assay_chembl_id),float(r.pIC50))
                              for r in g.itertuples(index=False)]
print('Unique molecular features:',len(STRUCTURES))

CHEMBL_NEAREST={}
COVERAGE=[]
for iso in ISOFORMS:
    lib=EXTERNAL_BY_ISO[iso]
    sample=ALL_LABELS[ALL_LABELS.isoform.eq(iso)].canonical_smiles.unique().tolist()
    for s in sample:
        near=nearest_cached(s,lib,FP_CACHE,top_k=1)
        sim=float(near[0][0]) if near else 0.
        ext_smi=near[0][1][0] if near else ''
        CHEMBL_NEAREST[(s,iso)]=(sim,ext_smi)
        COVERAGE.append({'canonical_smiles':s,'isoform':iso,
            'max_external_similarity':sim,'nearest_chembl_smiles':ext_smi})
COVERAGE=pd.DataFrame(COVERAGE)
COVERAGE.to_csv(OUT/'chembl_openadmet_coverage.csv',index=False)
COVER_SUM=[]
for iso,g in COVERAGE.groupby('isoform'):
    COVER_SUM.append({'isoform':iso,'n_openadmet_labeled':len(g),
      'n_chembl_external_compounds':EXTERNAL[EXTERNAL.isoform==iso].canonical_smiles.nunique(),
      **{f'fraction_sim_ge_{x:.2f}':float((g.max_external_similarity>=x).mean())
           for x in [.4,.5,.6,.7,.75,.8]},
      'median_nearest_similarity':float(g.max_external_similarity.median())})
COVERAGE_SUMMARY=pd.DataFrame(COVER_SUM)
COVERAGE_SUMMARY.to_csv(OUT/'chembl_coverage_by_isoform.csv',index=False)
display(COVERAGE_SUMMARY)

ASSAY_DISAGREE=(ASSAY_MEASUREMENTS.groupby(['isoform','canonical_smiles'])
    .agg(n_assays=('assay_chembl_id','nunique'),
         cross_assay_span=('pIC50',lambda x:float(x.max()-x.min())),
         cross_assay_median=('pIC50','median')).reset_index())
ASSAY_DISAGREE=ASSAY_DISAGREE[ASSAY_DISAGREE.n_assays>=2].copy()
ASSAY_DISAGREE.to_csv(OUT/'chembl_cross_assay_disagreement.csv',index=False)
print('Molecule/isoform combinations measured in ≥2 ChEMBL assays:',len(ASSAY_DISAGREE))

if BLIND_SMILES_CSV:
    blind=blind_struct.copy()
    assert 'SMILES' in blind
    if any('pIC50' in c or 'IC50' in c for c in blind.columns):
        raise ValueError('Blind structure audit must contain no activity labels.')
    audits=[]
    for s in blind.SMILES.map(canonical).dropna().unique():
        if s not in FP_CACHE:FP_CACHE[s]=fp_for(s,GEN)
        for iso in ISOFORMS:
            near=nearest_cached(s,EXTERNAL_BY_ISO[iso],FP_CACHE,1)
            audits.append({'canonical_smiles':s,'isoform':iso,
                   'nearest_external_similarity':near[0][0] if near else 0.})
    pd.DataFrame(audits).to_csv(OUT/'BLIND_STRUCTURE_ONLY_similarity_audit.csv',index=False)

## 7 · External delta model, assay calibration, and shuffled-label control

Within each **ChEMBL assay**, construct neighboring compound pairs; train a regularized antisymmetric Ridge model to predict `pIC50(query) − pIC50(anchor)` from molecular feature differences. The model cannot learn a fixed assay offset from the pairwise target. The same external training dataset is used for all outer folds because it contains **no exact OpenADMET structures**.

The calibrated ChEMBL kNN variant obtains offsets only from **training-fold exact overlaps** and is disabled when those overlaps are insufficient. A negative control shuffles pIC50 labels *within ChEMBL assay*, preserving source-level distributions while breaking chemical SAR.

In [ ]:
DELTA_MODELS={};NEG_MODELS={};PAIRS=[];PAIR_AUDIT=[]
SHUFFLE_EXT=EXTERNAL.copy()
shuffle_rng=np.random.default_rng(SEED+51)
if RUN_SHUFFLE_CONTROL:
    # Explicitly preserve distributions within each (isoform, assay) group.
    for (_, _),ix in SHUFFLE_EXT.groupby(['isoform','assay_chembl_id']).groups.items():
        inds=list(ix)
        SHUFFLE_EXT.loc[inds,'pIC50'] = shuffle_rng.permutation(SHUFFLE_EXT.loc[inds,'pIC50'].to_numpy())
for iso in ISOFORMS:
    model,pairs=create_chmbl_delta_pairs(EXTERNAL,FP_CACHE,FEAT_CACHE,iso,
                       min_sim=MIN_PAIR_SIM,max_pairs=MAX_PAIRS_PER_ISO,seed=SEED)
    DELTA_MODELS[iso]=model
    pairs['isoform']=iso
    PAIRS.append(pairs)
    if RUN_SHUFFLE_CONTROL:
        negative,_=create_chmbl_delta_pairs(SHUFFLE_EXT,FP_CACHE,FEAT_CACHE,iso,
                       min_sim=MIN_PAIR_SIM,max_pairs=MAX_PAIRS_PER_ISO,seed=SEED)
    else: negative=None
    NEG_MODELS[iso]=negative
    PAIR_AUDIT.append({'isoform':iso,'n_pairs':len(pairs),'trained':model is not None,
          'negative_trained':negative is not None,'assays_in_pairs':pairs.assay_id.nunique() if len(pairs) else 0})
PAIR_AUDIT=pd.DataFrame(PAIR_AUDIT)
PAIR_AUDIT.to_csv(OUT/'chembl_pair_training_audit.csv',index=False)
pd.concat(PAIRS,ignore_index=True).to_csv(OUT/'chembl_within_assay_delta_training_pairs.csv',index=False)
display(PAIR_AUDIT)
if not PAIR_AUDIT.trained.any():
    print('WARNING: no within-assay pairs; delta models will be inactive.')

## 8 · Signature-guarded prediction caches and leakage-safe inner/outer evaluation

Each **inner** validation split uses only the other inner training compounds as measured OpenADMET anchors and as calibration overlaps. Each **outer** test fold uses only the outer training molecules. If there are too few quality external analogs, the correction shrinks automatically to zero. Candidate weights are chosen **only from inner OOF**, optimizing the same single-pass macro ST-RAE definition as Notebook 12. Outer CI/labels are used for evaluation only.

`FAST_MODE` subsamples the measured validation rows per CYP and runs only fold 0; no fast score should be interpreted scientifically.

In [ ]:
SETTINGS={
  'nb11_sha':NB11_SHA,'external_sha':sha256_frame(EXTERNAL,
      ['isoform','assay_chembl_id','canonical_smiles','pIC50']),
  'overlap_sha':sha256_frame(OVERLAP,
      ['isoform','assay_chembl_id','canonical_smiles','pIC50']),
  'seed':SEED,'fast':FAST_MODE,'fp_bits':N_FP_BITS,
  'min_pair_sim':MIN_PAIR_SIM,'min_neighbor_sim':MIN_NEIGHBOR_SIM,
  'pairs':MAX_PAIRS_PER_ISO,'max_eval_rows_per_iso':MAX_EVAL_ROWS_PER_ISO,
  'strengths':BLEND_STRENGTHS,'shuffle':RUN_SHUFFLE_CONTROL,
  'tag':EXPERIMENT_TAG,'blind_structure_sha':sha256_file(BLIND_SMILES_CSV) if BLIND_SMILES_CSV else None}
RUN_ID=hashlib.sha256(json.dumps(SETTINGS,sort_keys=True,default=str).encode()).hexdigest()[:20]
RUN_CACHE=CACHE/('model_cache_'+RUN_ID)
RUN_CACHE.mkdir(exist_ok=True)
(OUT/'run_configuration.json').write_text(json.dumps(SETTINGS,indent=2,default=str))
print('Validated cache signature:',RUN_ID)

EXT_NEIGHBOR_CACHE={}
LEAKAGE_AUDIT=[]
def compute_rows(query,training,stage,outer_fold,inner_fold=None):
    # Label use is restricted to the current anchor partition.
    assert not set(query.canonical_smiles)&set(training.canonical_smiles)
    assert not set(EXTERNAL.canonical_smiles)&ALL_PRIMARY
    assert not set(EXTERNAL.canonical_smiles)&BLIND_EXACT
    results=[]
    for iso in ISOFORMS:
        q=query[query.isoform.eq(iso)]
        t=training[training.isoform.eq(iso)]
        by_assay,offset,n_overlaps=make_calibration(OVERLAP,training,iso)
        LEAKAGE_AUDIT.append({'stage':stage,'fold':outer_fold,'inner_fold':inner_fold,
             'isoform':iso,'n_val_rows':len(q),'n_training_rows':len(t),
             'n_calibration_molecules':n_overlaps,
             'overlap_validation_and_train':len(set(q.canonical_smiles)&set(t.canonical_smiles)),
             'external_exact_primary_overlap':len(set(EXTERNAL.canonical_smiles)&ALL_PRIMARY)})
        calib=(by_assay,offset,n_overlaps)
        anchor_smis=t.canonical_smiles.tolist()
        training_index=(anchor_smis,[FP_CACHE[s] for s in anchor_smis],t.y_true.to_numpy(float))
        for r in q.itertuples(index=False):
            rec=calc_local_one(r.canonical_smiles,iso,t,DELTA_MODELS[iso],
                         EXTERNAL_BY_ISO,FP_CACHE,FEAT_CACHE,EXT_NEIGHBOR_CACHE,
                         calib,min_ext_sim=MIN_NEIGHBOR_SIM,training_index=training_index,
                         negative_model=NEG_MODELS[iso] if RUN_SHUFFLE_CONTROL else None)
            # Shuffled negative control uses exactly the same anchor neighborhood.
            rec.update({'fold':int(outer_fold),'inner_fold':inner_fold,'stage':stage,
                'canonical_smiles':r.canonical_smiles,'isoform':iso,
                'scaffold_group':SCAFFOLD_BY_SMILES[r.canonical_smiles],
                'y_true':float(r.y_true),'baseline':float(r.baseline),
                'ci_low':float(r.ci_low),'ci_high':float(r.ci_high),
                'negative_delta_local':rec['negative_delta_local']})
            results.append(rec)
    return add_candidate_predictions(pd.DataFrame(results),BLEND_STRENGTHS)

def choose_fast_sample(f):
    if MAX_EVAL_ROWS_PER_ISO is None: return f
    return (f.groupby('isoform',group_keys=False)
             .apply(lambda g:g.sample(n=min(len(g),MAX_EVAL_ROWS_PER_ISO),random_state=SEED),include_groups=False)
             .reset_index().rename(columns={'index':'_index'}))

SELECTIONS=[];INNER_SCORES=[];OUTER_PRED=[]
for outer_fold in FOLDS_TO_RUN:
    inner_cache=RUN_CACHE/f'inner_{outer_fold}.csv'
    outer_cache=RUN_CACHE/f'outer_{outer_fold}.csv'
    inner_audit_cache=RUN_CACHE/f'inner_{outer_fold}_audit.json'
    outer_audit_cache=RUN_CACHE/f'outer_{outer_fold}_audit.json'
    if inner_cache.exists() and inner_audit_cache.exists():
        inner_pred=pd.read_csv(inner_cache)
        LEAKAGE_AUDIT.extend(json.loads(inner_audit_cache.read_text()))
        print('Resumed inner fold',outer_fold)
    else:
        start_audit=len(LEAKAGE_AUDIT)
        one=INNER[INNER.fold.eq(outer_fold)].copy()
        preds=[]
        for h in sorted(one.inner_fold.unique()):
            val=one[one.inner_fold.eq(h)].copy()
            train=one[one.inner_fold.ne(h)].copy()
            if MAX_EVAL_ROWS_PER_ISO is not None:
                val=(val.groupby('isoform',group_keys=False)
                         .sample(n=min(MAX_EVAL_ROWS_PER_ISO,val.groupby('isoform').size().min()),random_state=SEED+int(h)))
            preds.append(compute_rows(val,train,'inner',outer_fold,int(h)))
        inner_pred=pd.concat(preds,ignore_index=True).sort_values(['isoform','canonical_smiles','inner_fold'])
        inner_pred.to_csv(inner_cache,index=False,float_format='%.16g')
        inner_audit_cache.write_text(json.dumps(LEAKAGE_AUDIT[start_audit:],default=str))
        inner_pred=pd.read_csv(inner_cache)
    scores=scored_candidates(inner_pred,BLEND_STRENGTHS)
    scores['outer_fold']=outer_fold
    INNER_SCORES.append(scores)
    selected=str(scores.iloc[0].candidate)
    SELECTIONS.append({'outer_fold':outer_fold,'selected_candidate':selected,
        'inner_macro_st_rae':float(scores.iloc[0].macro_st_rae),
        'inner_macro_mae':float(scores.iloc[0].macro_mae),
        'inner_baseline_st_rae':float(scores[scores.candidate.eq('baseline')].macro_st_rae.iloc[0])})
    if outer_cache.exists() and outer_audit_cache.exists():
        outer_pred=pd.read_csv(outer_cache)
        LEAKAGE_AUDIT.extend(json.loads(outer_audit_cache.read_text()))
        print('Resumed outer fold',outer_fold)
    else:
        start_audit=len(LEAKAGE_AUDIT)
        val=OUTER[OUTER.fold.eq(outer_fold)].copy()
        train=INNER[INNER.fold.eq(outer_fold)].copy()
        if MAX_EVAL_ROWS_PER_ISO is not None:
            val=val.groupby('isoform',group_keys=False).sample(n=min(MAX_EVAL_ROWS_PER_ISO,val.groupby('isoform').size().min()),random_state=SEED)
        outer_pred=compute_rows(val,train,'outer',outer_fold)
        outer_pred.to_csv(outer_cache,index=False,float_format='%.16g')
        outer_audit_cache.write_text(json.dumps(LEAKAGE_AUDIT[start_audit:],default=str))
        outer_pred=pd.read_csv(outer_cache)
    outer_pred['selected_candidate']=selected
    outer_pred['selected']=outer_pred[selected]
    outer_pred['delta_fixed_0.30']=outer_pred['delta_0.30']
    outer_pred['calib_fixed_0.30']=outer_pred['calib_0.30']
    outer_pred['joint_fixed_0.30']=outer_pred['joint_0.30']
    outer_pred['negative_fixed_0.30']=(outer_pred.baseline+
        .3*outer_pred.delta_gate*np.where(np.isfinite(outer_pred.negative_delta_local),
                       outer_pred.negative_delta_local-outer_pred.baseline,0))
    OUTER_PRED.append(outer_pred)
    print('Outer fold',outer_fold,'nested selection:',selected,
          '| observed:',len(outer_pred),'| inner ST-RAE:',round(scores.iloc[0].macro_st_rae,5))

PRED=pd.concat(OUTER_PRED,ignore_index=True)
SELECTED=pd.DataFrame(SELECTIONS)
INNER_SCORES=pd.concat(INNER_SCORES,ignore_index=True)
if len(LEAKAGE_AUDIT):
    audit_df=pd.DataFrame(LEAKAGE_AUDIT)
    assert audit_df.overlap_validation_and_train.eq(0).all()
    assert audit_df.groupby(['fold','stage','isoform']).size().ge(1).all()
    assert pd.DataFrame(LEAKAGE_AUDIT).external_exact_primary_overlap.eq(0).all()
    pd.DataFrame(LEAKAGE_AUDIT).to_csv(OUT/'fit_leakage_audit.csv',index=False)
else:
    # Previous audits remain available on fully cached reruns.
    assert (OUT/'fit_leakage_audit.csv').is_file(), 'Missing cached leakage audit'
SELECTED.to_csv(OUT/'nested_selected_external_corrections.csv',index=False)
INNER_SCORES.to_csv(OUT/'inner_correction_candidate_scores.csv',index=False)
PRED.to_csv(OUT/'nested_outer_external_local_predictions.csv',index=False,float_format='%.15g')
assert PRED.canonical_smiles.notna().all()
assert (PRED.selected.to_numpy(float)==PRED.selected.to_numpy(float)).all()

## 9 · Model ablation, bootstrap, and similarity-conditioned diagnostics

Compare the untouched Notebook 11 baseline, a nested-selected external correction, two fixed local methods, their conservative joint correction, and a shuffled-ChEMBL control. All metrics are calculated on **identical observed compounds**. Report similarity-conditioned behavior without choosing new hyperparameters using these outer-fold labels.

In [ ]:
METRIC_COLS=['baseline','selected','delta_fixed_0.30','calib_fixed_0.30',
             'joint_fixed_0.30','negative_fixed_0.30']
rows=[]
for m in METRIC_COLS:
    rows.append({'model':m,'macro_mae':metric_mae(PRED,m),
                'macro_st_rae':st_rae(PRED,m),
                'n_evaluated':len(PRED),
                'fraction_active_correction':float((PRED[m]-PRED.baseline).abs().gt(1e-8).mean())})
SUMMARY=pd.DataFrame(rows).sort_values('macro_st_rae')
SUMMARY.to_csv(OUT/'model_ablation_summary.csv',index=False)
display(SUMMARY.round(6))
per=[]
for iso,g in PRED.groupby('isoform'):
    for model in METRIC_COLS:
        per.append({'isoform':iso,'model':model,
            'mae':float(abs(g.y_true-g[model]).mean()),
            'st_rae':float(np.sum(np.maximum(g.ci_low-g[model],0)+np.maximum(g[model]-g.ci_high,0)) /
                 np.sum(np.maximum(g.ci_low-g.y_true.mean(),0)+np.maximum(g.y_true.mean()-g.ci_high,0)))})
PER_ISO=pd.DataFrame(per)
PER_ISO.to_csv(OUT/'model_ablation_by_isoform.csv',index=False)

PRED['similarity_bin']=pd.cut(PRED.ext_max_sim,[-.01,.4,.55,.7,.8,1.01],
  labels=['<0.40','0.40–0.55','0.55–0.70','0.70–0.80','≥0.80'])
SIM_DIAG=(PRED.groupby(['isoform','similarity_bin'],observed=True)
   .apply(lambda g:pd.Series({'n':len(g),'baseline_mae':abs(g.baseline-g.y_true).mean(),
     'selected_mae':abs(g.selected-g.y_true).mean(),
     'improvement':abs(g.baseline-g.y_true).mean()-abs(g.selected-g.y_true).mean(),
     'mean_primary_similarity':g.primary_max_sim.mean()}),include_groups=False)
   .reset_index())
SIM_DIAG.to_csv(OUT/'performance_by_chembl_similarity.csv',index=False)

if len(FOLDS_TO_RUN)==5 and not FAST_MODE:
    boot=paired_group_bootstrap(PRED,baseline='baseline',candidate='selected',
                                reps=N_BOOTSTRAP,seed=SEED)
    boot.to_csv(OUT/'paired_scaffold_bootstrap.csv',index=False)
    print('Paired scaffold bootstrap MA-ST-RAE gain 95% interval:',
         boot.st_rae_gain.quantile([.025,.975]).to_dict())
else:
    print('Skipping full bootstrap: FAST_MODE or incomplete folds.')

print('External correction selected by fold:')
display(SELECTED)
print('Coverage (fraction with similarity ≥0.75):')
display(COVERAGE_SUMMARY[['isoform','fraction_sim_ge_0.75','median_nearest_similarity']])

## 10 · Export reproducible results, provenance and completeness metadata

This notebook **never generates a blinded-test submission**. A claim of improvement requires `FAST_MODE=False`, all five folds, complete ChEMBL retrieval, real Notebook 11 predictions, an appropriately positive nested score comparison, and inspection of potential data-provenance overlaps. Externally retrieved assay measurements should be disclosed in challenge submissions as required.

In [ ]:
META={'notebook':'15_chembl_augmented_local_cyp_qsar.ipynb',
      'retrieval_complete':RETRIEVAL_COMPLETE,
      'fast_mode':FAST_MODE,'folds_run':FOLDS_TO_RUN,
      'full_scientific_evaluation':bool(not FAST_MODE and MAX_EVAL_ROWS_PER_ISO is None
                    and RETRIEVAL_COMPLETE and set(FOLDS_TO_RUN)==set(range(5))
                    and (not CHEMBL_RAW_OVERRIDE or CHEMBL_SOURCE_COMPLETE_CONFIRMED)),
      'nb11_sha256':NB11_SHA,'nb11_real_chemeleon':bool(NB11_META.get('scientifically_valid_full_model_run')),
      'chembl_source':'user_csv' if CHEMBL_RAW_OVERRIDE else 'ChEMBL REST API',
      'chembl_api_url':API,'human_cyp_targets':TARGET_IDS,
      'cache_signature':RUN_ID,'settings':SETTINGS,
      'n_external_unique_structures':int(EXTERNAL.canonical_smiles.nunique()),
      'n_assay_level_external_measurements':len(EXTERNAL),
      'n_exact_overlap_quarantined':len(OVERLAP),
      'n_blind_exact_matches_quarantined':len(BLIND_QUARANTINE),
      'baseline_reference':'Notebook 11 nested CheMeleon_finetuned + ScreenStack_LGBM',
      'selection_rule':'Inner OOF macro ST-RAE; outer-fold labels used for evaluation only',
      'limitations':['Cross-assay IC50 values may not be directly comparable',
                     'Pretrained model training-data overlap remains unaudited',
                     'ChEMBL vintage and possible challenge-origin assays require manual review',
                     'This benchmark tests retrospective cross-validation, not blinded-test scoring']}
(OUT/'run_metadata.json').write_text(json.dumps(META,indent=2,default=str))
archive=BASE/'cyp_notebook_15_results.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for f in sorted(OUT.iterdir()):
        if f.is_file():z.write(f,'cyp_notebook_15_results/'+f.name)
print('Result ZIP:',archive,'size MiB',round(archive.stat().st_size/1048576,2))
print('SCIENTIFICALLY COMPLETE:',META['full_scientific_evaluation'])
print('Files:',*[f.name for f in sorted(OUT.iterdir()) if f.is_file()],sep='\n - ')